# Bab 04 — Kode yang diekstrak dari Word Master R53

Sumber: `MNUP-Manuskrip-Gabungan-Revision53-Repository-Code-QA-Candidate.docx`, rentang paragraf 1815–2581. Isi blok dipertahankan sesuai sumber; pseudocode, keluaran, dan fragmen yang bukan Python tidak diubah menjadi kode buatan. Jalankan per blok sesuai kebutuhan dan urutan buku.

**Blok tabel sumber 1** · diekstrak persis dari tabel Word Master R53; bab 4.

In [ ]:
import numpy as np


def gaussian_elimination(A, b, tol=1e-14):
    A = np.array(A, dtype=float, copy=True)
    b = np.array(b, dtype=float, copy=True)

    if A.ndim != 2 or A.shape[0] != A.shape[1]:
        raise ValueError("Matriks A harus persegi.")

    n = A.shape[0]
    if b.ndim != 1 or len(b) != n:
        raise ValueError("Dimensi b harus sesuai dengan A.")

    augmented = np.column_stack((A, b))

    for k in range(n - 1):
        # Partial pivoting: pilih elemen terbesar pada kolom aktif.
        pivot_row = k + np.argmax(np.abs(augmented[k:, k]))

        if abs(augmented[pivot_row, k]) < tol:
            raise ValueError("Matriks singular atau hampir singular.")

        if pivot_row != k:
            augmented[[k, pivot_row]] = augmented[[pivot_row, k]]

        pivot = augmented[k, k]
        for i in range(k + 1, n):
            factor = augmented[i, k] / pivot
            augmented[i, k:] -= factor * augmented[k, k:]

    # Pivot terakhir juga harus diperiksa sebelum substitusi balik.
    if abs(augmented[n - 1, n - 1]) < tol:
        raise ValueError("Matriks singular atau hampir singular.")

    x = np.zeros(n)
    for i in range(n - 1, -1, -1):
        if abs(augmented[i, i]) < tol:
            raise ValueError("Matriks singular atau hampir singular.")
        rhs = augmented[i, -1]
        known = np.dot(augmented[i, i + 1:n], x[i + 1:n])
        x[i] = (rhs - known) / augmented[i, i]

    return x

**Blok tabel sumber 2** · diekstrak persis dari tabel Word Master R53; bab 4.

In [ ]:
A = np.array([
    [4, 1, 1],
    [2, 5, 1],
    [1, 1, 3]
], dtype=float)

b = np.array([7, 13, 9], dtype=float)

x = gaussian_elimination(A, b)

print(x)
print("Residual:", np.linalg.norm(A @ x - b, ord=np.inf))

**Blok tabel sumber 3** · diekstrak persis dari tabel Word Master R53; bab 4.

In [ ]:
def gauss_jordan(A, b, tol=1e-14):
    A = np.array(A, dtype=float, copy=True)
    b = np.array(b, dtype=float, copy=True)

    n = len(b)
    augmented = np.column_stack((A, b))

    for k in range(n):
        # Partial pivoting
        pivot_row = k + np.argmax(
            np.abs(augmented[k:, k])
        )

        if abs(augmented[pivot_row, k]) < tol:
            raise ValueError("Matriks singular atau hampir singular.")

        if pivot_row != k:
            augmented[[k, pivot_row]] = augmented[[pivot_row, k]]

        # Normalisasi baris pivot
        augmented[k] /= augmented[k, k]

        # Eliminasi seluruh baris lain
        for i in range(n):
            if i == k:
                continue

            factor = augmented[i, k]
            augmented[i] -= factor * augmented[k]

    return augmented[:, -1]

**Blok tabel sumber 4** · diekstrak persis dari tabel Word Master R53; bab 4.

In [ ]:
x_gj = gauss_jordan(A, b)

print(x_gj)
print("Residual:",
      np.linalg.norm(A @ x_gj - b, ord=np.inf))

**Blok tabel sumber 5** · diekstrak persis dari tabel Word Master R53; bab 4.

In [ ]:
def lu_factorization(A, tol=1e-14):
    A = np.array(A, dtype=float)

    n = A.shape[0]
    L = np.eye(n)
    U = np.zeros_like(A)

    for i in range(n):

        for k in range(i, n):
            U[i, k] = (
                A[i, k]
                - np.dot(L[i, :i], U[:i, k])
            )

        if abs(U[i, i]) < tol:
            raise ValueError(
                "Pivot LU terlalu kecil."
            )

        for k in range(i + 1, n):
            U_ki = (
                A[k, i]
                - np.dot(L[k, :i], U[:i, i])
            )

            L[k, i] = U_ki / U[i, i]

    return L, U

**Blok tabel sumber 6** · diekstrak persis dari tabel Word Master R53; bab 4.

In [ ]:
def forward_substitution(L, b):
    n = len(b)
    y = np.zeros(n)

    for i in range(n):
        y[i] = (
            b[i] - np.dot(L[i, :i], y[:i])
        ) / L[i, i]

    return y


def backward_substitution(U, y):
    n = len(y)
    x = np.zeros(n)

    for i in range(n - 1, -1, -1):
        x[i] = (
            y[i]
            - np.dot(U[i, i + 1:], x[i + 1:])
        ) / U[i, i]

    return x

**Blok tabel sumber 7** · diekstrak persis dari tabel Word Master R53; bab 4.

In [ ]:
L, U = lu_factorization(A)

y = forward_substitution(L, b)
x_lu = backward_substitution(U, y)

print("L =\n", L)
print("U =\n", U)
print("x =", x_lu)
print("Residual:",
      np.linalg.norm(A @ x_lu - b, ord=np.inf))

**Blok tabel sumber 8** · diekstrak persis dari tabel Word Master R53; bab 4.

In [ ]:
def jacobi(A, b, x0=None,
           eps_x=1e-8,
           eps_f=1e-8,
           max_iter=100):

    A = np.array(A, dtype=float)
    b = np.array(b, dtype=float)

    n = len(b)

    if x0 is None:
        x = np.zeros(n)
    else:
        x = np.array(x0, dtype=float)

    history = []

    diagonal = np.diag(A)

    if np.any(np.abs(diagonal) < 1e-14):
        raise ValueError(
            "Terdapat elemen diagonal terlalu kecil."
        )

    for k in range(1, max_iter + 1):

        x_new = np.zeros(n)

        for i in range(n):
            sigma = (
                np.dot(A[i, :i], x[:i])
                + np.dot(A[i, i + 1:], x[i + 1:])
            )

            x_new[i] = (
                b[i] - sigma
            ) / A[i, i]

        delta = np.linalg.norm(x_new - x, ord=np.inf)
        residual = np.linalg.norm(A @ x_new - b, ord=np.inf)
        history.append(
            (k, x_new.copy(), delta, residual)
        )

        if delta <= eps_x and residual <= eps_f:
            return x_new, history, "converged"

        x = x_new

    return x, history, "maximum iteration reached"

**Blok tabel sumber 9** · diekstrak persis dari tabel Word Master R53; bab 4.

In [ ]:
def gauss_seidel(A, b, x0=None,
                 eps_x=1e-8,
                 eps_f=1e-8,
                 max_iter=100):

    A = np.array(A, dtype=float)
    b = np.array(b, dtype=float)

    n = len(b)

    if x0 is None:
        x = np.zeros(n)
    else:
        x = np.array(x0, dtype=float)

    history = []

    if np.any(np.abs(np.diag(A)) < 1e-14):
        raise ValueError(
            "Terdapat elemen diagonal terlalu kecil."
        )

    for k in range(1, max_iter + 1):

        x_old = x.copy()

        for i in range(n):

            lower = np.dot(
                A[i, :i],
                x[:i]
            )

            upper = np.dot(
                A[i, i + 1:],
                x_old[i + 1:]
            )

            x[i] = (b[i] - lower - upper) / A[i, i]
        delta = np.linalg.norm(x - x_old, ord=np.inf)
        residual = np.linalg.norm(A @ x - b, ord=np.inf)
        history.append(
            (k, x.copy(), delta, residual)
        )

        if delta <= eps_x and residual <= eps_f:
            return x, history, "converged"

    return x, history, "maximum iteration reached"

**Blok tabel sumber 10** · diekstrak persis dari tabel Word Master R53; bab 4.

In [ ]:
x_validation = np.linalg.solve(A, b)

print(x_validation)

**Blok tabel sumber 11** · diekstrak persis dari tabel Word Master R53; bab 4.

In [ ]:
import matplotlib.pyplot as plt

x_j, hist_j, status_j = jacobi(A, b)
x_gs, hist_gs, status_gs = gauss_seidel(A, b)

res_j = [item[3] for item in hist_j]
res_gs = [item[3] for item in hist_gs]

plt.figure(figsize=(7, 4))

plt.semilogy(
    range(1, len(res_j) + 1),
    res_j,
    marker="o",
    label="Jacobi"
)

plt.semilogy(
    range(1, len(res_gs) + 1),
    res_gs,
    marker="s",
    label="Gauss-Seidel"
)

plt.xlabel("Iterasi")
plt.ylabel(r"$||r||_\infty$")
plt.title("Konvergensi Metode Iteratif")
plt.grid(True)
plt.legend()
plt.show()